In [ ]:
import requests

# Beethoven file
candidates = [
    'https://api.github.com/repos/craigsapp/beethoven-piano-sonatas/contents/kern',
    'https://api.github.com/repos/craigsapp/beethoven-string-quartets/contents/kern',
]

for url in candidates:
    r = requests.get(url, timeout=10)
    if r.status_code == 200:
        files = [f['name'] for f in r.json()]
        print(f"✓ {url}")
        print(f"  {len(files)} files: {files[:5]}")
    else:
        print(f"✗ {url}  status={r.status_code}")

# KernScores
r = requests.get('https://api.github.com/search/repositories?q=beethoven+kern+music', timeout=10)
if r.status_code == 200:
    for repo in r.json()['items'][:5]:
        print(f"  repo: {repo['full_name']}")

✓ https://api.github.com/repos/craigsapp/beethoven-piano-sonatas/contents/kern
  103 files: ['sonata01-1.krn', 'sonata01-2.krn', 'sonata01-3.krn', 'sonata01-4.krn', 'sonata02-1.krn']
✓ https://api.github.com/repos/craigsapp/beethoven-string-quartets/contents/kern
  73 files: ['.kernscores', '.ref', 'quartet01-1.krn', 'quartet01-2.krn', 'quartet01-3.krn']


In [ ]:
import requests, os, time
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'
os.makedirs(OUT, exist_ok=True)

BASE1 = 'https://raw.githubusercontent.com/craigsapp/beethoven-piano-sonatas/master/kern'
BASE2 = 'https://raw.githubusercontent.com/craigsapp/beethoven-string-quartets/master/kern'

# file list
r1 = requests.get('https://api.github.com/repos/craigsapp/beethoven-piano-sonatas/contents/kern').json()
r2 = requests.get('https://api.github.com/repos/craigsapp/beethoven-string-quartets/contents/kern').json()

files1 = [(f['name'], BASE1) for f in r1 if f['name'].endswith('.krn')]
files2 = [(f['name'], BASE2) for f in r2 if f['name'].endswith('.krn') and f['name'].startswith('quartet')]

print(f"Downloading {len(files1)} sonata + {len(files2)} quartet files...")

for fname, base in files1 + files2:
    fpath = os.path.join(OUT, fname)
    if os.path.exists(fpath):
        continue
    r = requests.get(f'{base}/{fname}', timeout=15)
    if r.status_code == 200:
        with open(fpath, 'wb') as f:
            f.write(r.content)
    time.sleep(0.2)

print(f"Done. Files saved to {OUT}")

Mounted at /content/drive
Done. Files saved to /content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Filter Beethoven KRN files: 3/4 time only → compute per-piece σ
# ══════════════════════════════════════════════════════════════════

import os
import numpy as np
from music21 import converter

BEETHOVEN_DIR = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'

def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def is_three_four(score):
    for el in score.flatten():
        if hasattr(el, 'ratioString'):
            return el.ratioString == '3/4'
    return False

files = sorted([f for f in os.listdir(BEETHOVEN_DIR)
                if f.endswith('.krn')])

print(f"Total files: {len(files)}\n")
print(f"  {'File':<30} {'3/4':>5}  {'N meas':>7}  {'σ':>7}")
print(f"  {'─'*55}")

valid        = []
all_hists    = []
piece_sigmas = []

for fname in files:
    fpath = os.path.join(BEETHOVEN_DIR, fname)
    try:
        score = converter.parse(fpath)
        if not is_three_four(score):
            continue

        hists = []
        for part in score.parts:
            for m in part.getElementsByClass('Measure'):
                notes = list(m.flatten().notes)
                if len(notes) < 2:
                    continue
                h = pitch_class_histogram(m)
                if h.sum() > 0:
                    hists.append(h)
                    all_hists.append(h)

        s = sigma_of_hists(hists)
        if s is not None:
            piece_sigmas.append(s)
            valid.append(fname)
            print(f"  {fname:<30} {'✓':>5}  {len(hists):>7}  {s:>7.3f}")

    except Exception as e:
        print(f"  ERR {fname}: {e}")

print(f"\n{'='*55}")
print(f"  3/4 movements found: {len(valid)}")
print(f"  Total measures:      {len(all_hists)}")
if piece_sigmas:
    print(f"\n  Per-piece avg σ: {np.mean(piece_sigmas):.3f}  SD={np.std(piece_sigmas):.3f}")
    print(f"  Pooled σ:        {sigma_of_hists(all_hists):.3f}")
    print(f"\n  Mozart K.516f:   0.822  (paper)")

Total files: 174

  File                             3/4   N meas        σ
  ───────────────────────────────────────────────────────
  quartet01-1.krn                    ✓      743    1.380
  quartet01-3.krn                    ✓      344    1.452
  quartet02-2.krn                    ✓      294    1.164
  quartet02-3.krn                    ✓      256    1.315
  quartet03-3.krn                    ✓      320    1.411
  quartet04-3.krn                    ✓      272    1.481
  quartet05-2.krn                    ✓      244    1.383
  quartet06-3.krn                    ✓      229    1.354
  quartet08-3.krn                    ✓      423    1.390
  quartet09-1.krn                    ✓      727    1.305
  quartet09-3.krn                    ✓      315    1.201
  quartet10-3.krn                    ✓     1096    1.505
  quartet11-3.krn                    ✓      481    1.383
  quartet12-3.krn                    ✓     1262    1.412
  quartet13-1.krn                    ✓      807    1.261
  quartet13-

humdrum.spineParser: WARNING: Error in parsing event ('>') at position 1223 for spine None: Could not parse > for note information
humdrum.spineParser: WARNING: Error in parsing event (']') at position 1225 for spine None: Could not parse ] for note information
humdrum.spineParser: WARNING: Error in parsing event ('>') at position 1229 for spine None: Could not parse > for note information
humdrum.spineParser: WARNING: Error in parsing event (']') at position 1231 for spine None: Could not parse ] for note information
humdrum.spineParser: WARNING: Error in parsing event ('<') at position 1240 for spine None: Could not parse < for note information
humdrum.spineParser: WARNING: Error in parsing event ('p') at position 1250 for spine None: Could not parse p for note information
humdrum.spineParser: WARNING: Error in parsing event ('<') at position 1305 for spine None: Could not parse < for note information
humdrum.spineParser: WARNING: Error in parsing event ('p') at position 1315 for spi

  sonata18-1.krn                     ✓      215    1.284
  sonata18-3.krn                     ✓      134    0.994
  sonata20-2.krn                     ✓      226    1.152
  sonata22-1.krn                     ✓      285    1.070
  sonata25-1.krn                     ✓      355    1.262


humdrum.spineParser: WARNING: Error in parsing event ('4ryy4G- 4d-') at position 726 for spine None: 'Rest' object has no attribute 'pitch'


  sonata27-1.krn                     ✓      342    1.342
  sonata29-2.krn                     ✓      337    1.286


humdrum.spineParser: WARNING: Error in parsing event ('*clefG2yy') at position 83 for spine None: Unknown clef type *clefG2yy found


  sonata30-3.krn                     ✓      415    0.989
  sonata31-1.krn                     ✓      227    1.071

  3/4 movements found: 48
  Total measures:      17589

  Per-piece avg σ: 1.250  SD=0.159
  Pooled σ:        1.467

  Mozart K.516f:   0.822  (paper)


In [ ]:

# ══════════════════════════════════════════════════════════════════
# Beethoven: KRN → MIDI + full analysis (σ, Raw VL, Tymo VL, G-PELT)
# Replace Bach in corpus comparison
# ══════════════════════════════════════════════════════════════════

!pip install music21 ruptures -q

import os
import numpy as np
import ruptures as rpt
from music21 import converter, midi

from google.colab import drive
drive.mount('/content/drive')

KRN_DIR  = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'
MIDI_DIR = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_3_4'
os.makedirs(MIDI_DIR, exist_ok=True)

# Paper constants
SIGMA_MOZART  = 0.822
RAW_VL_MOZART = 2.143
TYMO_MOZART   = 0.508
GPELT_MOZART  = 0.019

# ── Helpers ───────────────────────────────────────────────────────
def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def compute_raw_vl(m):
    pitches = [n.pitch.midi for n in m.flatten().notes if hasattr(n, 'pitch')]
    if len(pitches) < 2:
        return 0.0
    return sum(abs(pitches[i+1]-pitches[i])
               for i in range(len(pitches)-1)) / len(pitches)

def compute_tymo_vl(m):
    pitches = [n.pitch.midi for n in m.flatten().notes if hasattr(n, 'pitch')]
    if len(pitches) < 2:
        return 0.0
    deltas = [abs((pitches[i] % 12) - (pitches[i+1] % 12))
              for i in range(len(pitches)-1)]
    return sum(min(d, 12-d) for d in deltas) / len(pitches)

def gpelt_density(mean_pitches, pen=3):
    if len(mean_pitches) < 6:
        return 0.0
    signal = np.array(mean_pitches).reshape(-1, 1)
    try:
        algo   = rpt.Pelt(model='rbf', min_size=2).fit(signal)
        breaks = algo.predict(pen=pen)
        return len([b for b in breaks if b < len(mean_pitches)]) / len(mean_pitches)
    except:
        return 0.0

def is_three_four(score):
    for el in score.flatten():
        if hasattr(el, 'ratioString'):
            return el.ratioString == '3/4'
    return False

# ── Convert KRN → MIDI + analyze ─────────────────────────────────
files = sorted([f for f in os.listdir(KRN_DIR) if f.endswith('.krn')])

all_hists       = []
all_raw_vls     = []
all_tymo_vls    = []
all_mean_pitches = []
piece_sigmas    = []
valid_files     = []

print(f"Processing {len(files)} files...\n")
print(f"  {'File':<30} {'N':>5}  {'σ':>6}  {'RawVL':>7}  {'TymoVL':>8}")
print(f"  {'─'*60}")

for fname in files:
    fpath = os.path.join(KRN_DIR, fname)
    try:
        score = converter.parse(fpath)
        if not is_three_four(score):
            continue

        # Save MIDI
        midi_name = fname.replace('.krn', '.mid')
        midi_path = os.path.join(MIDI_DIR, midi_name)
        score.write('midi', fp=midi_path)

        hists, raw_vls, tymo_vls, mean_pitches = [], [], [], []

        for part in score.parts:
            for m in part.getElementsByClass('Measure'):
                notes = list(m.flatten().notes)
                if len(notes) < 2:
                    continue
                h = pitch_class_histogram(m)
                if h.sum() > 0:
                    hists.append(h)
                    all_hists.append(h)
                raw_vls.append(compute_raw_vl(m))
                tymo_vls.append(compute_tymo_vl(m))
                pitches = [n.pitch.midi for n in notes if hasattr(n, 'pitch')]
                if pitches:
                    all_mean_pitches.append(np.mean(pitches))
                    mean_pitches.append(np.mean(pitches))

        all_raw_vls.extend(raw_vls)
        all_tymo_vls.extend(tymo_vls)

        s = sigma_of_hists(hists)
        if s is not None:
            piece_sigmas.append(s)
            valid_files.append(fname)
            print(f"  {fname:<30} {len(hists):>5}  {s:>6.3f}  "
                  f"{np.mean(raw_vls):>7.3f}  {np.mean(tymo_vls):>8.3f}")

    except Exception as e:
        pass

# ── Final results ─────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  {'Corpus':<25} {'σ':>6}  {'RawVL':>7}  {'TymoVL':>8}  {'GPELT':>7}")
print(f"  {'─'*60}")
print(f"  {'Mozart K.516f (paper)':<25} "
      f"{SIGMA_MOZART:>6.3f}  {RAW_VL_MOZART:>7.3f}  "
      f"{TYMO_MOZART:>8.3f}  {GPELT_MOZART:>7.3f}")

sigma_b = sigma_of_hists(all_hists)
raw_b   = np.mean(all_raw_vls)
tymo_b  = np.mean(all_tymo_vls)
gpelt_b = gpelt_density(all_mean_pitches)

print(f"  {'Beethoven (pooled)':<25} "
      f"{sigma_b:>6.3f}  {raw_b:>7.3f}  "
      f"{tymo_b:>8.3f}  {gpelt_b:>7.3f}")
print(f"  {'Beethoven (per-piece avg)':<25} "
      f"{np.mean(piece_sigmas):>6.3f}  {'─':>7}  {'─':>8}  {'─':>7}")
print(f"  {'─'*60}")
print(f"\n  N movements: {len(valid_files)}")
print(f"  N measures:  {len(all_hists)}")
print(f"\n  MIDI saved to: {MIDI_DIR}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing 174 files...

  File                               N       σ    RawVL    TymoVL
  ────────────────────────────────────────────────────────────
  quartet01-1.krn                  743   1.380    1.598     1.132
  quartet01-3.krn                  344   1.452    2.634     1.034
  quartet02-2.krn                  294   1.164    1.985     1.419
  quartet02-3.krn                  256   1.315    2.028     1.542
  quartet03-3.krn                  320   1.411    1.447     1.253
  quartet04-3.krn                  272   1.481    1.783     1.165
  quartet05-2.krn                  244   1.383    1.926     1.412
  quartet06-3.krn                  229   1.354    2.234     1.179
  quartet08-3.krn                  423   1.390    1.730     1.295
  quartet09-1.krn                  727   1.305

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Extract Beethoven Trio sections from quartet 3rd movements
# Trio = section after the Scherzo repeat (middle section)
# ══════════════════════════════════════════════════════════════════

import os
import numpy as np
import ruptures as rpt
from music21 import converter, stream

from google.colab import drive
drive.mount('/content/drive')

KRN_DIR  = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'
TRIO_DIR = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_trio'
os.makedirs(TRIO_DIR, exist_ok=True)

# Only quartet 3rd movements (Scherzo movements)
files = sorted([f for f in os.listdir(KRN_DIR)
                if f.startswith('quartet') and f.endswith('.krn')
                and '-3.krn' in f])

print(f"Scherzo movements found: {len(files)}\n")

def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def extract_trio(score):
    """
    Extract Trio section from a Scherzo movement.
    Strategy: find rehearsal marks or tempo marks labeled 'Trio',
    or split by key-change boundary at midpoint.
    """
    trio_measures = []
    in_trio = False

    for el in score.flatten():
        # Look for text expression 'Trio'
        if hasattr(el, 'content') and 'Trio' in str(el.content):
            in_trio = True
        if hasattr(el, 'text') and 'Trio' in str(el.text):
            in_trio = True

    if not in_trio:
        # Fallback: use middle third of the movement as Trio
        all_measures = []
        for part in score.parts[:1]:
            for m in part.getElementsByClass('Measure'):
                all_measures.append(m)
        n = len(all_measures)
        trio_measures = all_measures[n//3 : 2*n//3]
        return trio_measures, 'fallback'

    # Extract measures after Trio marker
    for part in score.parts[:1]:
        found = False
        for m in part.getElementsByClass('Measure'):
            for el in m.flatten():
                if (hasattr(el, 'content') and 'Trio' in str(el.content)) or \
                   (hasattr(el, 'text') and 'Trio' in str(el.text)):
                    found = True
            if found:
                trio_measures.append(m)
    return trio_measures, 'marker'

all_hists    = []
piece_sigmas = []

print(f"  {'File':<25} {'Method':<10} {'N meas':>7}  {'sigma':>7}")
print(f"  {'─'*55}")

for fname in files:
    fpath = os.path.join(KRN_DIR, fname)
    try:
        score       = converter.parse(fpath)
        trio_m, method = extract_trio(score)

        if len(trio_m) < 4:
            print(f"  {fname:<25} {'skip':>10}  too short")
            continue

        # Save as MIDI
        trio_score = stream.Score()
        trio_part  = stream.Part()
        for m in trio_m:
            trio_part.append(m)
        trio_score.append(trio_part)
        midi_name = fname.replace('.krn', '_trio.mid')
        trio_score.write('midi', fp=os.path.join(TRIO_DIR, midi_name))

        # Analyze
        hists = []
        for m in trio_m:
            notes = list(m.flatten().notes)
            if len(notes) < 2:
                continue
            h = pitch_class_histogram(m)
            if h.sum() > 0:
                hists.append(h)
                all_hists.append(h)

        s = sigma_of_hists(hists)
        if s is not None:
            piece_sigmas.append(s)
            print(f"  {fname:<25} {method:<10} {len(hists):>7}  {s:>7.3f}")

    except Exception as e:
        print(f"  ERR {fname}: {e}")

print(f"\n{'='*55}")
print(f"  Beethoven Trio pooled sigma:     {sigma_of_hists(all_hists):.3f}")
print(f"  Beethoven Trio per-piece avg:    {np.mean(piece_sigmas):.3f}")
print(f"  N movements: {len(piece_sigmas)}")
print(f"  N measures:  {len(all_hists)}")
print(f"\n  Saved to: {TRIO_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scherzo movements found: 16

  File                      Method      N meas    sigma
  ───────────────────────────────────────────────────────
  quartet01-3.krn           fallback        33    1.369
  quartet02-3.krn           fallback        21    1.181
  quartet03-3.krn           fallback        35    1.280
  quartet04-3.krn           fallback        28    1.275
  quartet05-3.krn           fallback        36    1.177
  quartet06-3.krn           fallback        22    1.166
  quartet07-3.krn           fallback        40    1.190
  quartet08-3.krn           fallback        21    1.097
  quartet09-3.krn           fallback        30    1.153
  quartet10-3.krn           fallback       105    1.478
  quartet11-3.krn           fallback        61    1.425
  quartet12-3.krn           fallback       126    1.395
  quartet13-3.krn           fallback        30    0.945


In [ ]:
# Check KRN file structure for Trio markers
KRN_DIR = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'

fname = 'quartet01-3.krn'
with open(os.path.join(KRN_DIR, fname), 'r') as f:
    lines = f.readlines()

# Print all non-note lines (markers, text, tempo, etc.)
for i, line in enumerate(lines):
    if not line.strip().startswith('!'):
        stripped = line.strip()
        if stripped and not all(c in '0123456789.-=[]()/ \t' for c in stripped):
            print(f"{i:4d}: {stripped}")

  13: **kern	**dynam	**kern	**dynam	**kern	**dynam	**kern	**dynam
  14: *staff4	*staff4	*staff3	*staff3	*staff2	*staff2	*staff1	*staff1
  15: *Icello	*Icello	*Iviola	*Iviola	*Ivioln	*Ivioln	*Ivioln	*Ivioln
  16: *>[A,A,B,B,C,C,D,A,B]	*>[A,A,B,B,C,C,D,A,B]	*>[A,A,B,B,C,C,D,A,B]	*>[A,A,B,B,C,C,D,A,B]	*>[A,A,B,B,C,C,D,A,B]	*>[A,A,B,B,C,C,D,A,B]	*>[A,A,B,B,C,C,D,A,B]	*>[A,A,B,B,C,C,D,A,B]
  17: *>norep[A,B,C,D,A,B]	*>norep[A,B,C,D,A,B]	*>norep[A,B,C,D,A,B]	*>norep[A,B,C,D,A,B]	*>norep[A,B,C,D,A,B]	*>norep[A,B,C,D,A,B]	*>norep[A,B,C,D,A,B]	*>norep[A,B,C,D,A,B]
  18: *>A	*>A	*>A	*>A	*>A	*>A	*>A	*>A
  19: *clefF4	*clefF4	*clefC3	*clefC3	*clefG2	*clefG2	*clefG2	*clefG2
  20: *k[b-]	*k[b-]	*k[b-]	*k[b-]	*k[b-]	*k[b-]	*k[b-]	*k[b-]
  21: *F:	*F:	*F:	*F:	*F:	*F:	*F:	*F:
  22: *M3/4	*M3/4	*M3/4	*M3/4	*M3/4	*M3/4	*M3/4	*M3/4
  23: *MM320	*MM320	*MM320	*MM320	*MM320	*MM320	*MM320	*MM320
  24: 4FF'/	p	2.A/	p	[2.c/	p	[2.f/	p
  25: 4AA'/	.	.	.	.	.	.	.
  26: 4BB-'/	.	.	.	.	.	.	.
  28: (2C/	.	2G/	.	2c/]	

In [ ]:
import os
import numpy as np
from music21 import converter, stream
from google.colab import drive

drive.mount('/content/drive')

KRN_DIR  = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'
TRIO_DIR = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_trio'
os.makedirs(TRIO_DIR, exist_ok=True)

files = sorted([f for f in os.listdir(KRN_DIR)
                if f.startswith('quartet') and '-3.krn' in f])

def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def extract_trio_from_krn(fpath):
    """
    Parse KRN file directly to find Trio section markers (*>C or *>D).
    Returns list of measure line blocks between Trio markers.
    """
    with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    # Find Trio section labels from header
    # *>[A,A,B,B,C,C,D,A,B] → Trio = C (and optionally D)
    trio_labels = set()
    for line in lines[:30]:
        if '*>[' in line or '*>norep[' in line:
            # Extract section sequence
            import re
            match = re.search(r'\[(.*?)\]', line)
            if match:
                sections = match.group(1).split(',')
                # Find unique labels: first appearance of non-A/B label
                seen = set()
                scherzo = []
                for s in sections:
                    s = s.strip()
                    if s not in seen:
                        seen.add(s)
                        scherzo.append(s)
                # Trio = sections after first 2 unique labels
                unique = list(dict.fromkeys(sections))
                if len(unique) >= 3:
                    trio_labels = set(unique[2:4])  # C, D typically
                break

    if not trio_labels:
        trio_labels = {'C', 'D'}  # default

    # Extract lines between Trio section markers
    in_trio   = False
    trio_lines = []
    header    = []
    in_header = True

    for line in lines:
        stripped = line.strip()
        # Collect header (spine setup lines)
        if in_header:
            if stripped.startswith('*>') and not stripped.startswith('*>[') and not stripped.startswith('*>norep'):
                label = stripped.split('\t')[0].replace('*>', '')
                if label in trio_labels:
                    in_trio   = True
                    in_header = False
                    trio_lines.append(line)
                else:
                    in_header = False
            else:
                header.append(line)
            continue

        if stripped.startswith('*>') and not stripped.startswith('*>[') and not stripped.startswith('*>norep'):
            label = stripped.split('\t')[0].replace('*>', '')
            if label in trio_labels:
                in_trio = True
            else:
                in_trio = False

        if in_trio:
            trio_lines.append(line)

    return header, trio_lines

all_hists    = []
piece_sigmas = []

print(f"Found {len(files)} Scherzo movements\n")
print(f"  {'File':<25} {'Trio labels':>12}  {'N':>5}  {'sigma':>7}")
print(f"  {'─'*55}")

for fname in files:
    fpath = os.path.join(KRN_DIR, fname)
    try:
        header, trio_lines = extract_trio_from_krn(fpath)

        if len(trio_lines) < 10:
            print(f"  {fname:<25} {'no trio':>12}")
            continue

        # Write temp KRN file and parse with music21
        tmp_path = f'/tmp/trio_{fname}'
        with open(tmp_path, 'w') as f:
            f.writelines(header + trio_lines + ['*-\t*-\t*-\t*-\t*-\t*-\t*-\t*-\n'])

        score = converter.parse(tmp_path)

        # Save MIDI
        midi_name = fname.replace('.krn', '_trio.mid')
        score.write('midi', fp=os.path.join(TRIO_DIR, midi_name))

        # Analyze
        hists = []
        for part in score.parts:
            for m in part.getElementsByClass('Measure'):
                notes = list(m.flatten().notes)
                if len(notes) < 2:
                    continue
                h = pitch_class_histogram(m)
                if h.sum() > 0:
                    hists.append(h)
                    all_hists.append(h)

        s = sigma_of_hists(hists)
        if s is not None:
            piece_sigmas.append(s)
            print(f"  {fname:<25} {'C,D':>12}  {len(hists):>5}  {s:>7.3f}")

    except Exception as e:
        print(f"  ERR {fname}: {e}")

print(f"\n{'='*55}")
print(f"  Beethoven Trio pooled sigma:   {sigma_of_hists(all_hists):.3f}")
print(f"  Beethoven Trio per-piece avg:  {np.mean(piece_sigmas):.3f}")
print(f"  N movements: {len(piece_sigmas)}")
print(f"  N measures:  {len(all_hists)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 16 Scherzo movements

  File                       Trio labels      N    sigma
  ───────────────────────────────────────────────────────
  ERR quartet01-3.krn: Could not determine spineType for spine with id 8
  quartet02-3.krn                    C,D     93    1.226
  ERR quartet03-3.krn: Could not determine spineType for spine with id 8
  quartet04-3.krn                no trio
  quartet05-3.krn                    C,D     51    1.097
  quartet06-3.krn                    C,D      6    0.000
  quartet07-3.krn                no trio
  quartet08-3.krn                    C,D     16    1.436
  quartet09-3.krn                    C,D     52    0.918
  quartet10-3.krn                    C,D    135    1.277
  ERR quartet11-3.krn: Could not determine spineType for spine with id 8
  quartet12-3.krn                    C,D      8    0.647
  quartet13-3.krn           

In [ ]:
KRN_DIR = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'

for fname in ['quartet04-3.krn', 'quartet07-3.krn', 'quartet13-3.krn']:
    fpath = os.path.join(KRN_DIR, fname)
    print(f"\n=== {fname} ===")
    with open(fpath, 'r', errors='ignore') as f:
        for i, line in enumerate(f):
            if i > 40:
                break
            stripped = line.strip()
            if stripped.startswith('*>'):
                print(f"  {i:4d}: {stripped}")


=== quartet04-3.krn ===
    16: *>[A,A,B,B1,B,B2,C,C,D,A,B,B2]	*>[A,A,B,B1,B,B2,C,C,D,A,B,B2]	*>[A,A,B,B1,B,B2,C,C,D,A,B,B2]	*>[A,A,B,B1,B,B2,C,C,D,A,B,B2]	*>[A,A,B,B1,B,B2,C,C,D,A,B,B2]	*>[A,A,B,B1,B,B2,C,C,D,A,B,B2]	*>[A,A,B,B1,B,B2,C,C,D,A,B,B2]	*>[A,A,B,B1,B,B2,C,C,D,A,B,B2]
    17: *>norep[A,B,B2,C,D,A,B,B2]	*>norep[A,B,B2,C,D,A,B,B2]	*>norep[A,B,B2,C,D,A,B,B2]	*>norep[A,B,B2,C,D,A,B,B2]	*>norep[A,B,B2,C,D,A,B,B2]	*>norep[A,B,B2,C,D,A,B,B2]	*>norep[A,B,B2,C,D,A,B,B2]	*>norep[A,B,B2,C,D,A,B,B2]
    18: *>A	*>A	*>A	*>A	*>A	*>A	*>A	*>A

=== quartet07-3.krn ===

=== quartet13-3.krn ===


In [ ]:
import os, re
import numpy as np
from music21 import converter
from google.colab import drive

drive.mount('/content/drive')

KRN_DIR  = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'
TRIO_DIR = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_trio'
os.makedirs(TRIO_DIR, exist_ok=True)

files = sorted([f for f in os.listdir(KRN_DIR)
                if f.startswith('quartet') and '-3.krn' in f])

def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def find_trio_labels(lines):
    """
    Find Trio section labels from KRN header.
    Strategy: from norep sequence, take unique labels,
    skip first two unique groups (Scherzo = A, B variants),
    rest = Trio labels.
    """
    for line in lines[:30]:
        if '*>norep[' in line:
            match = re.search(r'\[(.*?)\]', line)
            if match:
                sections = [s.strip() for s in match.group(1).split(',')]
                # Get unique labels preserving order
                seen   = []
                unique = []
                for s in sections:
                    base = re.sub(r'\d+$', '', s)  # strip trailing numbers
                    if base not in seen:
                        seen.append(base)
                        unique.append(s)
                # Scherzo = first 2 base groups, Trio = rest
                if len(unique) >= 3:
                    scherzo_bases = set(re.sub(r'\d+$', '', u) for u in unique[:2])
                    trio_labels   = set()
                    for s in sections:
                        base = re.sub(r'\d+$', '', s)
                        if base not in scherzo_bases:
                            trio_labels.add(s)
                    return trio_labels
    return set()

def extract_trio_lines(fpath):
    with open(fpath, 'r', errors='ignore') as f:
        lines = f.readlines()

    # Count spines from first data line
    n_spines = None
    for line in lines:
        if line.startswith('**'):
            n_spines = len(line.strip().split('\t'))
            break

    trio_labels = find_trio_labels(lines)
    if not trio_labels:
        return [], [], n_spines

    header     = []
    trio_lines = []
    in_header  = True
    in_trio    = False

    for line in lines:
        stripped = line.strip()

        if in_header:
            if (stripped.startswith('*>') and
                    not stripped.startswith('*>[') and
                    not stripped.startswith('*>norep')):
                label = stripped.split('\t')[0].replace('*>', '')
                in_header = False
                if label in trio_labels:
                    in_trio = True
                    trio_lines.append(line)
                else:
                    in_trio = False
            else:
                header.append(line)
            continue

        if (stripped.startswith('*>') and
                not stripped.startswith('*>[') and
                not stripped.startswith('*>norep')):
            label = stripped.split('\t')[0].replace('*>', '')
            in_trio = label in trio_labels

        if in_trio:
            trio_lines.append(line)

    return header, trio_lines, n_spines

all_hists    = []
piece_sigmas = []

print(f"Found {len(files)} Scherzo movements\n")
print(f"  {'File':<25} {'Trio labels':>15}  {'N':>5}  {'sigma':>7}")
print(f"  {'─'*60}")

for fname in files:
    fpath = os.path.join(KRN_DIR, fname)
    try:
        with open(fpath, 'r', errors='ignore') as f:
            raw = f.readlines()

        trio_labels = find_trio_labels(raw)
        header, trio_lines, n_spines = extract_trio_lines(fpath)

        if not trio_lines or len(trio_lines) < 10:
            print(f"  {fname:<25} {'no trio':>15}")
            continue

        # Write temp KRN
        tmp = f'/tmp/trio_{fname}'
        terminator = '\t'.join(['*-'] * (n_spines or 8)) + '\n'
        with open(tmp, 'w') as f:
            f.writelines(header + trio_lines + [terminator])

        score = converter.parse(tmp)
        score.write('midi',
            fp=os.path.join(TRIO_DIR, fname.replace('.krn', '_trio.mid')))

        hists = []
        for part in score.parts:
            for m in part.getElementsByClass('Measure'):
                notes = list(m.flatten().notes)
                if len(notes) < 2:
                    continue
                h = pitch_class_histogram(m)
                if h.sum() > 0:
                    hists.append(h)
                    all_hists.append(h)

        s = sigma_of_hists(hists)
        if s is not None:
            piece_sigmas.append(s)
        label_str = ','.join(sorted(trio_labels))
        print(f"  {fname:<25} {label_str:>15}  {len(hists):>5}  "
              f"{s:.3f}" if s else f"  {fname:<25} {'parsed, no sigma':>15}")

    except Exception as e:
        print(f"  ERR {fname}: {e}")

print(f"\n{'='*55}")
if all_hists:
    print(f"  Beethoven Trio pooled sigma:   {sigma_of_hists(all_hists):.3f}")
if piece_sigmas:
    print(f"  Beethoven Trio per-piece avg:  {np.mean(piece_sigmas):.3f}")
print(f"  N movements: {len(piece_sigmas)}")
print(f"  N measures:  {len(all_hists)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 16 Scherzo movements

  File                          Trio labels      N    sigma
  ────────────────────────────────────────────────────────────
  ERR quartet01-3.krn: Could not determine spineType for spine with id 8
  ERR quartet02-3.krn: Could not determine spineType for spine with id 8
  ERR quartet03-3.krn: Could not determine spineType for spine with id 8
  ERR quartet04-3.krn: Could not determine spineType for spine with id 8
  ERR quartet05-3.krn: Could not determine spineType for spine with id 12
  ERR quartet06-3.krn: Could not determine spineType for spine with id 8
  quartet07-3.krn                   no trio
  ERR quartet08-3.krn: Could not determine spineType for spine with id 8
  ERR quartet09-3.krn: Could not determine spineType for spine with id 8
  ERR quartet10-3.krn: Could not determine spineType for spine with id 8
  quartet11-3.krn 

In [ ]:
import os, re
import numpy as np
from music21 import converter, stream
from google.colab import drive

drive.mount('/content/drive')

KRN_DIR  = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_menuets'
TRIO_DIR = '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_trio'
os.makedirs(TRIO_DIR, exist_ok=True)

files = sorted([f for f in os.listdir(KRN_DIR)
                if f.startswith('quartet') and '-3.krn' in f])

def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def get_trio_labels(fpath):
    """Read KRN file and find Trio section labels."""
    with open(fpath, 'r', errors='ignore') as f:
        lines = f.readlines()
    for line in lines[:30]:
        if '*>norep[' in line:
            match = re.search(r'\[(.*?)\]', line)
            if match:
                sections = [s.strip() for s in match.group(1).split(',')]
                unique_bases = []
                seen = []
                for s in sections:
                    base = re.sub(r'\d+$', '', s)
                    if base not in seen:
                        seen.append(base)
                        unique_bases.append(base)
                # Trio = all labels whose base is not in first 2 unique bases
                scherzo_bases = set(unique_bases[:2])
                trio_labels   = set()
                for s in sections:
                    base = re.sub(r'\d+$', '', s)
                    if base not in scherzo_bases:
                        trio_labels.add(s)
                return trio_labels, lines
    return set(), lines

def get_trio_measure_range(lines, trio_labels):
    """
    Find measure numbers that belong to Trio sections
    by scanning raw KRN for section markers and barlines.
    Returns (start_measure, end_measure) or None.
    """
    in_trio      = False
    measure_num  = 0
    trio_measures = []

    for line in lines:
        stripped = line.strip()

        # Section marker
        if (stripped.startswith('*>') and
                not stripped.startswith('*>[') and
                not stripped.startswith('*>norep')):
            label = stripped.split('\t')[0].replace('*>', '')
            in_trio = label in trio_labels
            continue

        # Barline — increment measure counter
        if stripped.startswith('=') and not stripped.startswith('=='):
            try:
                num = int(re.search(r'=(\d+)', stripped).group(1))
                measure_num = num
            except:
                measure_num += 1

        if in_trio and measure_num > 0:
            trio_measures.append(measure_num)

    if not trio_measures:
        return None, None
    return min(trio_measures), max(trio_measures)

all_hists    = []
piece_sigmas = []

print(f"Found {len(files)} Scherzo movements\n")
print(f"  {'File':<25} {'Trio labels':>15}  {'Meas range':>12}  {'N':>5}  {'sigma':>7}")
print(f"  {'─'*70}")

for fname in files:
    fpath = os.path.join(KRN_DIR, fname)
    try:
        trio_labels, lines = get_trio_labels(fpath)

        if not trio_labels:
            print(f"  {fname:<25} {'no trio':>15}")
            continue

        m_start, m_end = get_trio_measure_range(lines, trio_labels)

        if m_start is None:
            print(f"  {fname:<25} {'no measures':>15}")
            continue

        # Parse full file with music21
        score = converter.parse(fpath)

        # Extract Trio measures by number
        hists = []
        trio_part = stream.Part()
        for part in score.parts[:1]:
            for m in part.getElementsByClass('Measure'):
                if m_start <= m.number <= m_end:
                    notes = list(m.flatten().notes)
                    if len(notes) < 2:
                        continue
                    h = pitch_class_histogram(m)
                    if h.sum() > 0:
                        hists.append(h)
                        all_hists.append(h)
                    trio_part.append(m)

        # Save MIDI
        trio_score = stream.Score()
        trio_score.append(trio_part)
        trio_score.write('midi',
            fp=os.path.join(TRIO_DIR, fname.replace('.krn', '_trio.mid')))

        s = sigma_of_hists(hists)
        if s is not None:
            piece_sigmas.append(s)
        label_str = ','.join(sorted(trio_labels))
        s_str     = f"{s:.3f}" if s else "N/A"
        print(f"  {fname:<25} {label_str:>15}  "
              f"{m_start}–{m_end:>6}  {len(hists):>5}  {s_str:>7}")

    except Exception as e:
        print(f"  ERR {fname}: {e}")

print(f"\n{'='*60}")
if all_hists:
    print(f"  Beethoven Trio pooled sigma:   {sigma_of_hists(all_hists):.3f}")
if piece_sigmas:
    print(f"  Beethoven Trio per-piece avg:  {np.mean(piece_sigmas):.3f}")
print(f"  N movements: {len(piece_sigmas)}")
print(f"  N measures:  {len(all_hists)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 16 Scherzo movements

  File                          Trio labels    Meas range      N    sigma
  ──────────────────────────────────────────────────────────────────────
  quartet01-3.krn                       C,D  86–   145     43    1.236
  quartet02-3.krn                  C,D,D2,E  44–    88     34    1.063
  quartet03-3.krn                       C,D  63–   168     67    1.349
  quartet04-3.krn                       C,D  52–    99     39    1.114
  quartet05-3.krn           C,D,E,F,G,H,I,I2,J,K,L,L2,M,M2  17–   142     96    1.141
  quartet06-3.krn                    C,D,D2  50–    70     20    0.909
  quartet07-3.krn                   no trio
  quartet08-3.krn                    B,B2,C  12–   141     84    1.331
  quartet09-3.krn               C,C2,D,D2,E  39–    97     51    1.271
  quartet10-3.krn                     C,D,E  78–   468    234    1.48

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Figure 1, 2, 3 — Bach replaced by Beethoven
# Run this entire cell in Colab
# ══════════════════════════════════════════════════════════════════

!pip install music21 ruptures umap-learn -q

import os
import numpy as np
import ruptures as rpt
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from music21 import converter
from google.colab import drive

drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────
PATHS = {
    'Mozart':     '/content/drive/MyDrive/MozartDiceGame/MIDI/mozart',
    'Haydn':      '/content/drive/MyDrive/MozartDiceGame/MIDI/haydn',
    'Beethoven':  '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_3_4',
    'Schubert':   '/content/drive/MyDrive/MozartDiceGame/MIDI/schubert_minuets',
}
TRIO_PATHS = {
    'Mozart':    '/content/drive/MyDrive/MozartDiceGame/MIDI/mozart',
    'Haydn':     '/content/drive/MyDrive/MozartDiceGame/MIDI/haydn_trio',
    'Beethoven': '/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_trio',
    'Schubert':  '/content/drive/MyDrive/MozartDiceGame/MIDI/schubert_trio',
}

OUT_DIR = '/content/drive/MyDrive/MozartDiceGame/figures_beethoven'
os.makedirs(OUT_DIR, exist_ok=True)

# Colors consistent with paper
COLORS = {
    'Mozart':    '#006BA4',
    'Haydn':     '#ABABAB',
    'Beethoven': '#595959',
    'Schubert':  '#7B4F2E',
}

# ── Helpers ───────────────────────────────────────────────────────
def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def top5_key_coverage(folder):
    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith('.mid') or
                       f.lower().endswith('.MID') or
                       f.lower().endswith('.krn')])
    counts = {}
    total  = 0
    for fname in files:
        try:
            score = converter.parse(os.path.join(folder, fname))
            k     = score.analyze('key')
            key   = f"{k.tonic} {k.mode}"
            counts[key] = counts.get(key, 0) + 1
            total += 1
        except:
            pass
    if total == 0:
        return 0.0
    top5 = sorted(counts.values(), reverse=True)[:5]
    return sum(top5) / total * 100

def load_corpus_vecs(folder):
    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith('.mid') or
                       f.lower().endswith('.MID') or
                       f.lower().endswith('.krn')])
    hists = []
    for fname in files:
        try:
            score = converter.parse(os.path.join(folder, fname))
            for part in score.parts:
                for m in part.getElementsByClass('Measure'):
                    if len(list(m.flatten().notes)) < 2:
                        continue
                    h = pitch_class_histogram(m)
                    if h.sum() > 0:
                        hists.append(h)
        except:
            pass
    return np.array(hists) if hists else np.zeros((0, 12))

def build_mean_pitch_signal(folder):
    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith('.mid') or
                       f.lower().endswith('.MID')])
    signals = []
    for fname in files:
        try:
            score = converter.parse(os.path.join(folder, fname))
            for part in score.parts:
                means = []
                for m in part.getElementsByClass('Measure'):
                    pitches = [n.pitch.midi for n in m.flatten().notes
                               if hasattr(n, 'pitch')]
                    if pitches:
                        means.append(np.mean(pitches))
            if means:
                signals.extend(means)
        except:
            pass
    return signals

def gpelt_density_from_signal(signal, pen=3):
    if len(signal) < 6:
        return 0.0
    arr = np.array(signal).reshape(-1, 1)
    try:
        algo   = rpt.Pelt(model='rbf', min_size=2).fit(arr)
        breaks = algo.predict(pen=pen)
        return len([b for b in breaks if b < len(signal)]) / len(signal)
    except:
        return 0.0

def gpelt_sensitivity(folder, penalties=[2,3,4,5]):
    signal = build_mean_pitch_signal(folder)
    return {pen: gpelt_density_from_signal(signal, pen) for pen in penalties}

# ══════════════════════════════════════════════════════════════════
# Load all data
# ══════════════════════════════════════════════════════════════════
print("Loading corpus data...")

menuet_coverage = {}
menuet_vecs     = {}
menuet_sigma    = {}
trio_coverage   = {}

for name, path in PATHS.items():
    print(f"  {name} Menuet...")
    vecs = load_corpus_vecs(path)
    menuet_vecs[name]     = vecs
    menuet_sigma[name]    = sigma_of_hists(list(vecs)) if len(vecs) > 1 else 0
    menuet_coverage[name] = top5_key_coverage(path)

for name, path in TRIO_PATHS.items():
    print(f"  {name} Trio...")
    try:
        trio_coverage[name] = top5_key_coverage(path)
    except:
        trio_coverage[name] = 0.0

print("\nCoverage (Menuet / Trio):")
for name in PATHS:
    print(f"  {name}: {menuet_coverage[name]:.1f}% / {trio_coverage.get(name,0):.1f}%")

print("\nSigma:")
for name in PATHS:
    print(f"  {name}: {menuet_sigma[name]:.3f}")

# ══════════════════════════════════════════════════════════════════
# FIGURE 1: Top-5 Key Coverage Bar Chart
# ══════════════════════════════════════════════════════════════════
print("\nGenerating Figure 1...")

corpora  = list(PATHS.keys())
x        = np.arange(len(corpora))
width    = 0.35

menuet_vals = [menuet_coverage[c] for c in corpora]
trio_vals   = [trio_coverage.get(c, 0) for c in corpora]

fig, ax = plt.subplots(figsize=(7, 4))

bars1 = ax.bar(x - width/2, menuet_vals, width,
               color=[COLORS[c] for c in corpora],
               label='Menuet', edgecolor='white', linewidth=0.8)
bars2 = ax.bar(x + width/2, trio_vals, width,
               color=[COLORS[c] for c in corpora],
               label='Trio', edgecolor='white', linewidth=0.8,
               hatch='///', alpha=0.7)

for bar, val in zip(list(bars1) + list(bars2),
                    menuet_vals + trio_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.8,
            f'{val:.0f}%',
            ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(corpora, fontsize=10)
ax.set_ylabel('Top-5 key coverage (%)', fontsize=10)
ax.set_ylim(0, 115)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(['Menuet', 'Trio'], fontsize=9, loc='upper right')
ax.set_title('Tonal concentration: top-5 key coverage by corpus',
             fontsize=10, fontweight='bold')

plt.tight_layout()
out1 = os.path.join(OUT_DIR, 'fig_tonal_homogeneity.png')
plt.savefig(out1, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"  Saved: {out1}")

# ══════════════════════════════════════════════════════════════════
# FIGURE 2: UMAP (Menuet + Trio side by side)
# ══════════════════════════════════════════════════════════════════
print("Generating Figure 2 (UMAP)...")

import umap

def run_umap(vecs_dict, max_per=500, seed=42):
    np.random.seed(seed)
    all_vecs   = []
    all_labels = []
    for name, vecs in vecs_dict.items():
        if len(vecs) == 0:
            continue
        idx = (np.random.choice(len(vecs), min(max_per, len(vecs)),
                                replace=False)
               if len(vecs) > max_per else np.arange(len(vecs)))
        all_vecs.append(vecs[idx])
        all_labels.extend([name] * len(idx))
    all_vecs = np.vstack(all_vecs)
    reducer  = umap.UMAP(n_neighbors=15, min_dist=0.1,
                         metric='manhattan', random_state=seed)
    proj     = reducer.fit_transform(all_vecs)
    return proj, all_labels

# Trio vecs
trio_vecs = {}
for name, path in TRIO_PATHS.items():
    try:
        trio_vecs[name] = load_corpus_vecs(path)
    except:
        trio_vecs[name] = np.zeros((0, 12))

proj_m, labels_m = run_umap(menuet_vecs)
proj_t, labels_t = run_umap(trio_vecs)

SIZES  = {'Mozart': 40, 'Haydn': 8, 'Beethoven': 8, 'Schubert': 8}
ALPHAS = {'Mozart': 0.9, 'Haydn': 0.3, 'Beethoven': 0.3, 'Schubert': 0.3}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, proj, labels, title in [
        (axes[0], proj_m, labels_m, '(a) Menuet'),
        (axes[1], proj_t, labels_t, '(b) Trio')]:
    for name in ['Haydn', 'Beethoven', 'Schubert', 'Mozart']:
        idx = [i for i, l in enumerate(labels) if l == name]
        if not idx:
            continue
        n = len(idx)
        ax.scatter(proj[idx, 0], proj[idx, 1],
                   c=COLORS[name], s=SIZES[name],
                   alpha=ALPHAS[name],
                   label=f'{name} (n={n})',
                   zorder=3 if name == 'Mozart' else 1)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('UMAP 1', fontsize=9)
    ax.set_ylabel('UMAP 2', fontsize=9)
    ax.legend(fontsize=8, markerscale=1.5)
    ax.grid(True, alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('UMAP projections of pitch-class distributions ($L^1$ metric)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
out2 = os.path.join(OUT_DIR, 'fig_umap_menuet_beethoven.png')
plt.savefig(out2, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"  Saved: {out2}")

# ══════════════════════════════════════════════════════════════════
# FIGURE 3: G-PELT Sensitivity
# ══════════════════════════════════════════════════════════════════
print("Generating Figure 3 (G-PELT)...")

penalties = [2, 3, 4, 5]

# Mozart: use generated pieces folder
MOZART_PIECES = '/content/drive/MyDrive/MozartDiceGame/MIDI/mozart_pieces_graphTEST'

gpelt_data = {}
for name, path in PATHS.items():
    p = MOZART_PIECES if name == 'Mozart' else path
    gpelt_data[name] = gpelt_sensitivity(p, penalties)
    print(f"  {name}: {gpelt_data[name]}")

fig, ax = plt.subplots(figsize=(6, 4))

for name in PATHS:
    vals = [gpelt_data[name][p] for p in penalties]
    ax.plot(penalties, vals,
            color=COLORS[name],
            linewidth=2.5 if name == 'Mozart' else 1.5,
            linestyle='-' if name == 'Mozart' else '--',
            marker='o', markersize=5,
            label=name,
            zorder=3 if name == 'Mozart' else 1)

ax.set_xlabel('Penalty $\\lambda$', fontsize=10)
ax.set_ylabel('G-PELT changepoint density', fontsize=10)
ax.set_title('G-PELT changepoint density vs.\ penalty',
             fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
out3 = os.path.join(OUT_DIR, 'fig_gpelt_beethoven.png')
plt.savefig(out3, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"  Saved: {out3}")

print(f"\nAll figures saved to: {OUT_DIR}")
print("\nFinal sigma values:")
for name in PATHS:
    print(f"  {name}: sigma = {menuet_sigma[name]:.3f}")

<>:319: SyntaxWarning: invalid escape sequence '\ '
<>:319: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_1824/3212103891.py:319: SyntaxWarning: invalid escape sequence '\ '
  ax.set_title('G-PELT changepoint density vs.\ penalty',


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading corpus data...
  Mozart Menuet...
  Haydn Menuet...
  Beethoven Menuet...
  Schubert Menuet...
  Mozart Trio...
  Haydn Trio...


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'All rights reserved by OnClassical.com, \xa9 2000'>; getting generic Instrument
  warnings.warn(


  Beethoven Trio...
  Schubert Trio...

Coverage (Menuet / Trio):
  Mozart: 84.6% / 84.6%
  Haydn: 76.0% / 75.0%
  Beethoven: 52.1% / 56.2%
  Schubert: 83.8% / 62.5%

Sigma:
  Mozart: 0.882
  Haydn: 1.511
  Beethoven: 1.458
  Schubert: 1.272

Generating Figure 1...
  Saved: /content/drive/MyDrive/MozartDiceGame/figures_beethoven/fig_tonal_homogeneity.png
Generating Figure 2 (UMAP)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Saved: /content/drive/MyDrive/MozartDiceGame/figures_beethoven/fig_umap_menuet_beethoven.png
Generating Figure 3 (G-PELT)...
  Mozart: {2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0}
  Haydn: {2: 0.04211041743175419, 3: 0.018851756640959727, 4: 0.012118986412045538, 5: 0.007467254253886645}
  Beethoven: {2: 0.040473840078973346, 3: 0.025830865416255348, 4: 0.017110891740704178, 5: 0.012010529779532741}
  Schubert: {2: 0.02028485110056107, 3: 0.013810962451445835, 4: 0.011221406991799741, 5: 0.008200258955545965}
  Saved: /content/drive/MyDrive/MozartDiceGame/figures_beethoven/fig_gpelt_beethoven.png

All figures saved to: /content/drive/MyDrive/MozartDiceGame/figures_beethoven

Final sigma values:
  Mozart: sigma = 0.882
  Haydn: sigma = 1.511
  Beethoven: sigma = 1.458
  Schubert: sigma = 1.272


In [ ]:
# Fix 1: Mozart sigma — M prefix only
PATHS['Mozart'] = '/content/drive/MyDrive/MozartDiceGame/MIDI/mozart'

def load_corpus_vecs_filtered(folder, prefix=None):
    files = sorted([f for f in os.listdir(folder)
                    if (f.lower().endswith('.mid') or f.upper().endswith('.MID'))
                    and (prefix is None or f.upper().startswith(prefix))])
    hists = []
    for fname in files:
        try:
            score = converter.parse(os.path.join(folder, fname))
            for part in score.parts:
                for m in part.getElementsByClass('Measure'):
                    if len(list(m.flatten().notes)) < 2:
                        continue
                    h = pitch_class_histogram(m)
                    if h.sum() > 0:
                        hists.append(h)
        except:
            pass
    return np.array(hists) if hists else np.zeros((0, 12))

# Mozart: M prefix only
vecs = load_corpus_vecs_filtered(PATHS['Mozart'], prefix='M')
print(f"Mozart M-only: {len(vecs)} measures, sigma = {sigma_of_hists(list(vecs)):.3f}")

# Beethoven coverage check
print(f"\nBeethoven top-5 coverage: {menuet_coverage['Beethoven']:.1f}%")
print("(48 movements across many keys — naturally lower)")

Mozart M-only: 176 measures, sigma = 0.822

Beethoven top-5 coverage: 52.1%
(48 movements across many keys — naturally lower)


In [ ]:
# Per-piece top-1 key coverage — average across pieces
import os
import numpy as np
from music21 import converter
from google.colab import drive

drive.mount('/content/drive')

PATHS = {
    'Mozart':    ('/content/drive/MyDrive/MozartDiceGame/MIDI/mozart', 'M'),
    'Haydn':     ('/content/drive/MyDrive/MozartDiceGame/MIDI/haydn', None),
    'Beethoven': ('/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_3_4', None),
    'Schubert':  ('/content/drive/MyDrive/MozartDiceGame/MIDI/schubert_minuets', None),
}
TRIO_PATHS = {
    'Mozart':    ('/content/drive/MyDrive/MozartDiceGame/MIDI/mozart', 'T'),
    'Haydn':     ('/content/drive/MyDrive/MozartDiceGame/MIDI/haydn_trio', None),
    'Beethoven': ('/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_trio', None),
    'Schubert':  ('/content/drive/MyDrive/MozartDiceGame/MIDI/schubert_trio', None),
}

def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def per_piece_top1_coverage(folder, prefix=None):
    """
    For each piece: compute fraction of measures whose
    detected key matches the piece's overall key.
    Return mean across pieces.
    """
    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith('.mid') or
                       f.upper().endswith('.MID') or
                       f.lower().endswith('.krn')])
    if prefix:
        files = [f for f in files if f.upper().startswith(prefix)]

    piece_coverages = []

    for fname in files:
        fpath = os.path.join(folder, fname)
        try:
            score    = converter.parse(fpath)
            piece_key = str(score.analyze('key').tonic) + ' ' + \
                        str(score.analyze('key').mode)

            key_counts = {}
            total      = 0
            for part in score.parts:
                for m in part.getElementsByClass('Measure'):
                    notes = list(m.flatten().notes)
                    if len(notes) < 2:
                        continue
                    try:
                        mk  = score.analyze('key')
                        mks = str(mk.tonic) + ' ' + str(mk.mode)
                        key_counts[mks] = key_counts.get(mks, 0) + 1
                        total += 1
                    except:
                        pass

            if total == 0:
                continue

            # Top-1 key fraction
            top1 = max(key_counts.values()) / total * 100
            piece_coverages.append(top1)

        except Exception as e:
            pass

    if not piece_coverages:
        return 0.0
    return float(np.mean(piece_coverages)), len(piece_coverages)

# Better approach: use pitch-class histograms directly
# For each piece, find dominant key via pitch-class and measure coverage
def per_piece_dominant_pc_coverage(folder, prefix=None):
    """
    For each piece: find the dominant pitch class (tonic),
    compute fraction of notes belonging to that major scale.
    Return mean across pieces.
    """
    MAJOR_SCALE = [0, 2, 4, 5, 7, 9, 11]  # intervals from tonic

    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith('.mid') or
                       f.upper().endswith('.MID') or
                       f.lower().endswith('.krn')])
    if prefix:
        files = [f for f in files if f.upper().startswith(prefix)]

    piece_coverages = []

    for fname in files:
        fpath = os.path.join(folder, fname)
        try:
            score = converter.parse(fpath)
            k     = score.analyze('key')
            tonic = k.tonic.pitchClass  # 0-11

            # Key PCs
            key_pcs = set((tonic + interval) % 12
                          for interval in MAJOR_SCALE)

            # Count measures in key
            in_key = 0
            total  = 0
            for part in score.parts:
                for m in part.getElementsByClass('Measure'):
                    notes = list(m.flatten().notes)
                    if len(notes) < 2:
                        continue
                    hist = pitch_class_histogram(m)
                    if hist.sum() == 0:
                        continue
                    # Fraction of note weight in key
                    in_key_frac = sum(hist[pc] for pc in key_pcs)
                    in_key += in_key_frac
                    total  += 1

            if total == 0:
                continue
            piece_coverages.append(in_key / total * 100)

        except:
            pass

    if not piece_coverages:
        return 0.0, 0
    return float(np.mean(piece_coverages)), len(piece_coverages)

print(f"Computing per-piece dominant-key coverage...\n")
print(f"  {'Corpus':<12} {'Menuet':>10}  {'N':>5}  {'Trio':>10}  {'N':>5}")
print(f"  {'─'*50}")

for name in ['Mozart', 'Haydn', 'Beethoven', 'Schubert']:
    folder, prefix = PATHS[name]
    m_cov, m_n    = per_piece_dominant_pc_coverage(folder, prefix)

    tfolder, tprefix = TRIO_PATHS[name]
    try:
        t_cov, t_n = per_piece_dominant_pc_coverage(tfolder, tprefix)
    except:
        t_cov, t_n = 0.0, 0

    print(f"  {name:<12} {m_cov:>10.1f}%  {m_n:>5}  {t_cov:>10.1f}%  {t_n:>5}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Computing per-piece dominant-key coverage...

  Corpus           Menuet      N        Trio      N
  ──────────────────────────────────────────────────
  Mozart             87.6%    176        91.5%     96
  Haydn              85.1%     75        86.1%     40
  Beethoven          79.5%     48        80.6%     16
  Schubert           82.2%     37        74.7%      8


In [ ]:
import os
import numpy as np
from music21 import converter
from google.colab import drive

drive.mount('/content/drive')

TRIO_PATHS = {
    'Haydn':    '/content/drive/MyDrive/MozartDiceGame/MIDI/haydn_trio',
    'Schubert': '/content/drive/MyDrive/MozartDiceGame/MIDI/schubert_trio',
}

def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def per_piece_sigma(folder):
    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith('.mid') or
                       f.upper().endswith('.MID') or
                       f.lower().endswith('.krn')])
    piece_sigmas = []
    all_hists    = []

    for fname in files:
        try:
            score = converter.parse(os.path.join(folder, fname))
            hists = []
            for part in score.parts:
                for m in part.getElementsByClass('Measure'):
                    if len(list(m.flatten().notes)) < 2:
                        continue
                    h = pitch_class_histogram(m)
                    if h.sum() > 0:
                        hists.append(h)
                        all_hists.append(h)
            s = sigma_of_hists(hists)
            if s is not None:
                piece_sigmas.append(s)
        except:
            pass

    return (sigma_of_hists(all_hists),
            float(np.mean(piece_sigmas)) if piece_sigmas else None,
            len(piece_sigmas))

print(f"  {'Corpus':<12} {'Pooled σ':>10}  {'Per-piece σ':>12}  {'N':>5}")
print(f"  {'─'*45}")

for name, path in TRIO_PATHS.items():
    pooled, avg, n = per_piece_sigma(path)
    print(f"  {name:<12} {pooled:>10.3f}  {avg:>12.3f}  {n:>5}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Corpus         Pooled σ   Per-piece σ      N
  ─────────────────────────────────────────────
  Haydn             1.390         1.287     40
  Schubert          1.488         1.338      8


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CORPORA = ['Mozart\nK.516f', 'Haydn', 'Beethoven', 'Schubert']
COLORS  = ['#006BA4', '#ABABAB', '#595959', '#7B4F2E']

MENUET_SIGMA = [0.822, 1.345, 1.250, 1.088]  # per-piece avg
TRIO_SIGMA   = [0.869, 1.287, 1.229, 1.338]  # per-piece avg

x     = np.arange(len(CORPORA))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4.5))

bars1 = ax.bar(x - width/2, MENUET_SIGMA, width,
               color=COLORS, label='Menuet',
               edgecolor='white', linewidth=0.8)
bars2 = ax.bar(x + width/2, TRIO_SIGMA, width,
               color=COLORS, label='Trio',
               edgecolor='white', linewidth=0.8,
               hatch='///', alpha=0.75)

for bar, val in zip(list(bars1) + list(bars2),
                    MENUET_SIGMA + TRIO_SIGMA):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(CORPORA, fontsize=10)
ax.set_ylabel('Corpus spread ($\\sigma$, per-piece avg)', fontsize=10)
ax.set_ylim(0, 1.75)
ax.axhline(y=0.822, color='#006BA4', linestyle='--',
           linewidth=1.2, alpha=0.5)
ax.axhline(y=0.869, color='#006BA4', linestyle=':',
           linewidth=1.2, alpha=0.5)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=9, loc='upper right')
ax.set_title(
    'Tonal homogeneity: pitch-class corpus spread ($\\sigma$)\n'
    'per-piece average, Menuet and Trio sections',
    fontsize=10, fontweight='bold')

plt.tight_layout()
OUT = '/content/drive/MyDrive/MozartDiceGame/figures_beethoven/fig_tonal_homogeneity.png'
plt.savefig(OUT, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {OUT}")

Saved: /content/drive/MyDrive/MozartDiceGame/figures_beethoven/fig_tonal_homogeneity.png


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Figures 1, 2, 3 — Fixed version
# - Fig 1: single baseline
# - Fig 2: Mozart M-only (n=176), distinct colors
# - Fig 3: title fix
# ══════════════════════════════════════════════════════════════════

!pip install music21 ruptures umap-learn -q

import os
import numpy as np
import ruptures as rpt
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from music21 import converter
from google.colab import drive

drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/MozartDiceGame/figures_beethoven'
os.makedirs(OUT_DIR, exist_ok=True)

COLORS = {
    'Mozart':    '#006BA4',
    'Haydn':     '#ABABAB',
    'Beethoven': '#444444',
    'Schubert':  '#7B4F2E',
}

MENUET_PATHS = {
    'Mozart':    ('/content/drive/MyDrive/MozartDiceGame/MIDI/mozart',            'M'),
    'Haydn':     ('/content/drive/MyDrive/MozartDiceGame/MIDI/haydn',             None),
    'Beethoven': ('/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_3_4',     None),
    'Schubert':  ('/content/drive/MyDrive/MozartDiceGame/MIDI/schubert_minuets',  None),
}
TRIO_PATHS = {
    'Mozart':    ('/content/drive/MyDrive/MozartDiceGame/MIDI/mozart',            'T'),
    'Haydn':     ('/content/drive/MyDrive/MozartDiceGame/MIDI/haydn_trio',        None),
    'Beethoven': ('/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_trio',    None),
    'Schubert':  ('/content/drive/MyDrive/MozartDiceGame/MIDI/schubert_trio',     None),
}

# ── Helpers ───────────────────────────────────────────────────────
def pitch_class_histogram(m):
    hist = np.zeros(12)
    for n in m.flatten().notes:
        if hasattr(n, 'pitch'):
            hist[n.pitch.midi % 12] += 1
        elif hasattr(n, 'pitches'):
            for p in n.pitches:
                hist[p.midi % 12] += 1
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def sigma_of_hists(hists):
    if len(hists) < 2:
        return None
    arr = np.array(hists)
    c   = arr.mean(axis=0)
    if c.sum() > 0:
        c /= c.sum()
    return float(np.mean([np.sum(np.abs(h - c)) for h in arr]))

def load_vecs(folder, prefix=None):
    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith('.mid') or
                       f.upper().endswith('.MID') or
                       f.lower().endswith('.krn')])
    if prefix:
        files = [f for f in files if f.upper().startswith(prefix)]
    hists = []
    for fname in files:
        try:
            score = converter.parse(os.path.join(folder, fname))
            for part in score.parts:
                for m in part.getElementsByClass('Measure'):
                    if len(list(m.flatten().notes)) < 2:
                        continue
                    h = pitch_class_histogram(m)
                    if h.sum() > 0:
                        hists.append(h)
        except:
            pass
    return np.array(hists) if hists else np.zeros((0, 12))

def build_signal(folder, prefix=None):
    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith('.mid') or
                       f.upper().endswith('.MID')])
    if prefix:
        files = [f for f in files if f.upper().startswith(prefix)]
    means = []
    for fname in files:
        try:
            score = converter.parse(os.path.join(folder, fname))
            for part in score.parts:
                for m in part.getElementsByClass('Measure'):
                    pitches = [n.pitch.midi for n in m.flatten().notes
                               if hasattr(n, 'pitch')]
                    if pitches:
                        means.append(np.mean(pitches))
        except:
            pass
    return means

def gpelt_density(signal, pen=3):
    if len(signal) < 6:
        return 0.0
    arr = np.array(signal).reshape(-1, 1)
    try:
        algo   = rpt.Pelt(model='rbf', min_size=2).fit(arr)
        breaks = algo.predict(pen=pen)
        return len([b for b in breaks if b < len(signal)]) / len(signal)
    except:
        return 0.0

# ── Load data ─────────────────────────────────────────────────────
print("Loading data...")
menuet_vecs = {}
trio_vecs   = {}

for name, (folder, prefix) in MENUET_PATHS.items():
    menuet_vecs[name] = load_vecs(folder, prefix)
    print(f"  {name} Menuet: {len(menuet_vecs[name])} measures")

for name, (folder, prefix) in TRIO_PATHS.items():
    try:
        trio_vecs[name] = load_vecs(folder, prefix)
        print(f"  {name} Trio:   {len(trio_vecs[name])} measures")
    except:
        trio_vecs[name] = np.zeros((0, 12))

# ══════════════════════════════════════════════════════════════════
# FIGURE 1: Sigma bar chart (per-piece avg)
# ══════════════════════════════════════════════════════════════════
print("\nFigure 1...")

CORPORA      = ['Mozart\nK.516f', 'Haydn', 'Beethoven', 'Schubert']
MENUET_SIGMA = [0.822, 1.345, 1.250, 1.088]
TRIO_SIGMA   = [0.869, 1.287, 1.229, 1.338]
COLORS_LIST  = [COLORS[k.split('\n')[0].split(' ')[0]]
                for k in ['Mozart', 'Haydn', 'Beethoven', 'Schubert']]

x     = np.arange(4)
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4.5))

bars1 = ax.bar(x - width/2, MENUET_SIGMA, width,
               color=COLORS_LIST, edgecolor='white',
               linewidth=0.8, label='Menuet')
bars2 = ax.bar(x + width/2, TRIO_SIGMA, width,
               color=COLORS_LIST, edgecolor='white',
               linewidth=0.8, hatch='///', alpha=0.75, label='Trio')

for bar, val in zip(list(bars1) + list(bars2),
                    MENUET_SIGMA + TRIO_SIGMA):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.axhline(y=0.822, color='#006BA4', linestyle='--',
           linewidth=1.2, alpha=0.5, label='Mozart Menuet floor')
ax.set_xticks(x)
ax.set_xticklabels(CORPORA, fontsize=10)
ax.set_ylabel('Corpus spread ($\\sigma$, per-piece avg)', fontsize=10)
ax.set_ylim(0, 1.75)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=9, loc='upper right')
ax.set_title('Tonal homogeneity: pitch-class corpus spread ($\\sigma$)',
             fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_tonal_homogeneity.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Saved fig_tonal_homogeneity.png")

# ══════════════════════════════════════════════════════════════════
# FIGURE 2: UMAP
# ══════════════════════════════════════════════════════════════════
print("Figure 2 (UMAP)...")

import umap

SIZES  = {'Mozart': 40, 'Haydn': 8, 'Beethoven': 8, 'Schubert': 8}
ALPHAS = {'Mozart': 0.9, 'Haydn': 0.3, 'Beethoven': 0.3, 'Schubert': 0.3}

def run_umap(vecs_dict, seed=42):
    np.random.seed(seed)
    all_vecs, all_labels = [], []
    for name, vecs in vecs_dict.items():
        if len(vecs) == 0:
            continue
        n   = min(500, len(vecs))
        idx = np.random.choice(len(vecs), n, replace=False)
        all_vecs.append(vecs[idx])
        all_labels.extend([name] * n)
    all_vecs = np.vstack(all_vecs)
    reducer  = umap.UMAP(n_neighbors=15, min_dist=0.1,
                         metric='manhattan', random_state=seed)
    return reducer.fit_transform(all_vecs), all_labels

proj_m, labels_m = run_umap(menuet_vecs)
proj_t, labels_t = run_umap(trio_vecs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, proj, labels, title in [
        (axes[0], proj_m, labels_m, '(a) Menuet'),
        (axes[1], proj_t, labels_t, '(b) Trio')]:
    for name in ['Haydn', 'Beethoven', 'Schubert', 'Mozart']:
        idx = [i for i, l in enumerate(labels) if l == name]
        if not idx:
            continue
        ax.scatter(proj[idx, 0], proj[idx, 1],
                   c=COLORS[name],
                   s=SIZES[name],
                   alpha=ALPHAS[name],
                   label=f'{name} (n={len(idx)})',
                   zorder=3 if name == 'Mozart' else 1)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('UMAP 1', fontsize=9)
    ax.set_ylabel('UMAP 2', fontsize=9)
    ax.legend(fontsize=8, markerscale=1.5)
    ax.grid(True, alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('UMAP projections of pitch-class distributions ($L^1$ metric)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_umap_beethoven.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Saved fig_umap_beethoven.png")

# ══════════════════════════════════════════════════════════════════
# FIGURE 3: G-PELT
# ══════════════════════════════════════════════════════════════════
print("Figure 3 (G-PELT)...")

MOZART_PIECES = '/content/drive/MyDrive/MozartDiceGame/MIDI/mozart_pieces_graphTEST'
penalties     = [2, 3, 4, 5]

gpelt_paths = {
    'Mozart':    (MOZART_PIECES, None),
    'Haydn':     ('/content/drive/MyDrive/MozartDiceGame/MIDI/haydn', None),
    'Beethoven': ('/content/drive/MyDrive/MozartDiceGame/MIDI/beethoven_3_4', None),
    'Schubert':  ('/content/drive/MyDrive/MozartDiceGame/MIDI/schubert_minuets', None),
}

gpelt_data = {}
for name, (folder, prefix) in gpelt_paths.items():
    signal = build_signal(folder, prefix)
    gpelt_data[name] = {p: gpelt_density(signal, p) for p in penalties}
    print(f"  {name}: {gpelt_data[name]}")

fig, ax = plt.subplots(figsize=(6, 4))

for name in ['Mozart', 'Haydn', 'Beethoven', 'Schubert']:
    vals = [gpelt_data[name][p] for p in penalties]
    ax.plot(penalties, vals,
            color=COLORS[name],
            linewidth=2.5 if name == 'Mozart' else 1.5,
            linestyle='-'  if name == 'Mozart' else '--',
            marker='o', markersize=5,
            label=name,
            zorder=3 if name == 'Mozart' else 1)

ax.set_xlabel('Penalty $\\lambda$', fontsize=10)
ax.set_ylabel('G-PELT changepoint density', fontsize=10)
ax.set_title('G-PELT changepoint density vs. penalty $\\lambda$',
             fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_gpelt_beethoven.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Saved fig_gpelt_beethoven.png")

print(f"\nAll done. Saved to: {OUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data...
  Mozart Menuet: 176 measures
  Haydn Menuet: 35455 measures
  Beethoven Menuet: 18429 measures
  Schubert Menuet: 3779 measures
  Mozart Trio:   96 measures
  Haydn Trio:   8231 measures
  Beethoven Trio:   968 measures
  Schubert Trio:   1661 measures

Figure 1...
  Saved fig_tonal_homogeneity.png
Figure 2 (UMAP)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Saved fig_umap_beethoven.png
Figure 3 (G-PELT)...
  Mozart: {2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0}
